# Chator Server External Test
Run this notebook in Google Colab to test the Chator server from outside.

**Set the server IP below** before running.

In [ ]:
# CONFIG — change this to your server's public IP or hostname
TARGET_HOST = "178.166.228.93"
TARGET_PORT_TCP = 3478
TARGET_PORT_UDP = 3478
TARGET_PORT_HTTP = 8008

print(f"Target: {TARGET_HOST}")
print(f"HTTP:    http://{TARGET_HOST}:{TARGET_PORT_HTTP}")
print(f"TURN:    {TARGET_HOST}:{TARGET_PORT_TCP} (TCP/UDP)")

## 1. Install dependencies

In [ ]:
!apt-get update -qq && apt-get install -y -qq nodejs npm netcat-openbsd 2>&1 | tail -5
!npm install -g yarn 2>&1 | tail -3
print("Dependencies installed")

In [ ]:
!mkdir -p /tmp/chator-test && cd /tmp/chator-test && npm init -y 2>&1 | tail -1
!cd /tmp/chator-test && npm install turn 2>&1 | tail -3
print("turn module installed")

## 2. Test Synapse HTTP API

In [ ]:
import urllib.request
import json

def http_test(path, label):
    url = f"http://{TARGET_HOST}:{TARGET_PORT_HTTP}{path}"
    try:
        r = urllib.request.urlopen(url, timeout=10)
        body = r.read().decode()
        print(f"✅ {label}: HTTP {r.status}")
        if body:
            print(f"   Response: {body[:200]}")
        return True
    except Exception as e:
        print(f"❌ {label}: {e}")
        return False

http_test("/health", "Synapse /health")
http_test("/_matrix/client/versions", "Synapse versions")

## 3. Test TURN TCP connection

In [ ]:
import socket

sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
sock.settimeout(10)
try:
    result = sock.connect_ex((TARGET_HOST, TARGET_PORT_TCP))
    if result == 0:
        print(f"✅ TURN TCP port {TARGET_PORT_TCP} is OPEN")
        # Try reading STUN response
        import struct
        # STUN binding request
        msg = struct.pack("!BBHHH", 0x00, 0x01, 0x00, 0x00, 0x00, 0x00) + b"\x00" * 16
        msg = msg[:20] + struct.pack("!I", 0x2112A442) + msg[24:]
        sock.send(msg)
        data = sock.recv(1024)
        if len(data) >= 20:
            print(f"✅ STUN response received ({len(data)} bytes)")
        else:
            print(f"⚠️  Connected but short response ({len(data)} bytes)")
    else:
        print(f"❌ TURN TCP port {TARGET_PORT_TCP} is CLOSED (code: {result})")
except Exception as e:
    print(f"❌ TURN TCP error: {e}")
finally:
    sock.close()

## 4. Test TURN UDP (STUN binding)

In [ ]:
import socket
import struct

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.settimeout(10)
try:
    # STUN binding request
    tid = b"\x00" * 12
    msg = struct.pack("!BBHHH", 0x00, 0x01, 0x00, 0x00, 0x00, 0x00) + tid
    msg = msg[:20] + struct.pack("!I", 0x2112A442) + msg[24:]
    
    sock.sendto(msg, (TARGET_HOST, TARGET_PORT_UDP))
    data, addr = sock.recvfrom(1024)
    print(f"✅ STUN/UDP response from {addr[0]}:{addr[1]} ({len(data)} bytes)")
    if len(data) >= 20:
        msg_type = struct.unpack("!BB", data[:2])
        print(f"   Message type: 0x{msg_type[0]:02x}{msg_type[1]:02x}")
    print(f"✅ TURN UDP port {TARGET_PORT_UDP} is OPEN and responding")
except Exception as e:
    print(f"❌ TURN UDP error: {e}")
finally:
    sock.close()

## 5. Full TURN Allocation Test (authenticated)

In [ ]:
import hashlib, hmac, time, struct

turn_secret = "chator-test-secret"
username = "chator_test"
timestamp = int(time.time()) + 3600
temp_username = f"{timestamp}:{username}"
password = hmac.new(turn_secret.encode(), temp_username.encode(), hashlib.sha1).hexdigest()

print(f"TURN username: {temp_username}")
print(f"TURN password: {password}")

# Build STUN TURN Allocate request
tid = bytes([1,2,3,4,5,6,7,8,9,0,1,2])
# TURN Allocate = 0x0003 (method=3, class=0x00=Request)
msg_type = struct.pack("!H", 0x0003)
# Length placeholder
msg_len = struct.pack("!H", 0)
# Magic cookie
cookie = struct.pack("!I", 0x2112A442)
header = msg_type + msg_len + cookie + tid

# Attributes: REQUESTED-TRANSPORT (0x0019) = UDP (17)
attr_type = struct.pack("!H", 0x0019)
attr_len = struct.pack("!H", 4)
attr_val = struct.pack("!B", 17) + b"\x00\x00\x00"
body = attr_type + attr_len + attr_val

# Update length
msg_len = struct.pack("!H", len(body))
request = msg_type + msg_len + cookie + tid + body

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.settimeout(10)
try:
    sock.sendto(request, (TARGET_HOST, TARGET_PORT_UDP))
    data, addr = sock.recvfrom(4096)
    print(f"✅ TURN Allocate response from {addr[0]}:{addr[1]} ({len(data)} bytes)")
    
    # Parse response
    r_msg_type = struct.unpack("!H", data[:2])[0]
    r_len = struct.unpack("!H", data[2:4])[0]
    r_cookie = struct.unpack("!I", data[4:8])[0]
    
    r_class = r_msg_type & 0x0110
    r_method = r_msg_type & 0x3EED
    
    print(f"   Response method: 0x{r_method:04x}, class: 0x{r_class:04x}")
    
    if r_class == 0x0100:
        print(f"✅ 401 Unauthorized (expected) — TURN requires auth!")
        print(f"   ➡️  Server is alive and responding to TURN requests")
    elif r_method == 0x0003 and r_class == 0x0000:
        print(f"✅ Allocate success!")
    else:
        print(f"⚠️  Unexpected response type")
except Exception as e:
    print(f"❌ TURN Allocate error: {e}")
finally:
    sock.close()

## Results
Copy the output above — it shows whether your Chator server is reachable from the outside internet.